In [17]:
import cvxpy as cp
import heapq
import itertools
import numpy as np
import pandas as pd
import time
import tqdm
import warnings

warnings.filterwarnings("ignore")

In [18]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors, dtype=float) / len(priors)
    return priors / np.sum(priors)

def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + ((len(search_space)-i)*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump], axis=1)
    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i) * 1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])
    return X_p

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true, return_breakdowns=False):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    if return_breakdowns:
        return np.dot(losses, posteriors), losses * priors
    else:
        return np.dot(losses, posteriors)

def evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns=False):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true, return_breakdowns)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c, return_breakdowns=False):
    acc_loss = 0.0
    acc_loss_list = []
    for partition in partitions:
        if return_breakdowns:
            acc_loss_p, acc_loss_p_list = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
            acc_loss_list.append(acc_loss_p_list.tolist())
        else:
            acc_loss_p = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    if return_breakdowns:
        return acc_loss, acc_loss_list
    else:
        return acc_loss

In [19]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    t0 = time.perf_counter()
    indices = list(range(len(thresholds)))
    best_partition, best_loss = None, np.inf
    for partition in set_partitions(indices):
        acc_loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
        if acc_loss <= best_loss:
            if best_loss - acc_loss < 1e-9:
                if len(partition) < len(best_partition):
                    best_loss = acc_loss
                    best_partition = partition
            else:
                best_loss = acc_loss
                best_partition = partition
    
    return best_partition, best_loss, time.perf_counter()-t0

In [20]:
def find_partitions_greedy_agg(X, thresholds, priors, threshold_true, c, eps=1e-9):
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_conditional(X, block, thresholds, priors, threshold_true, c)
                + evaluate_conditional(X_eps, block, thresholds, priors, threshold_true, c) * eps)
    t0 = time.perf_counter()
    P = {}
    next_id = 0
    for i in range(len(priors)):
        P[next_id] = [i]
        next_id += 1

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = block_cost(ab) * np.sum(priors[ab])
        acc_loss_a  = block_cost(a)  * np.sum(priors[a])
        acc_loss_b  = block_cost(b)  * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P or b_id not in P:
            continue
        a, b = P[a_id], P[b_id]
        ab = sorted(a + b)

        acc_loss_a  = block_cost(a)
        acc_loss_b  = block_cost(b)
        acc_loss_ab = block_cost(ab)

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]
            pq = [(g, (x, y)) for g, (x, y) in pq if x not in {a_id, b_id} and y not in {a_id, b_id}]
            heapq.heapify(pq)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P:
                if p_id == new_id:
                    continue
                p = P[p_id]
                merged = sorted(ab + p)
                acc_loss_merged = block_cost(merged) * np.sum(priors[merged])
                acc_loss_p      = block_cost(p) * np.sum(priors[p])
                acc_loss_ab_new = block_cost(ab) * np.sum(priors[ab])
                gain = -(acc_loss_p + acc_loss_ab_new - acc_loss_merged)
                heapq.heappush(pq, (gain, (new_id, p_id)))
    partition = list(P.values())
    loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
    return partition, loss, time.perf_counter()-t0

def split_partition(partition):
    idx_large = 0
    block_large = partition[idx_large]
    block_others = [partition[i] for i in range(len(partition)) if i != idx_large]
    result = []
    for part in itertools.combinations(block_large, len(block_large)-1):
        A = list(part)
        B = [x for x in block_large if x not in A]
        result.append([A] + [B] + block_others)

        for i, block in enumerate(block_others):
            merged = sorted(B + block)
            other_remaining = [block_others[j] for j in range(len(block_others)) if j != i]
            result.append([A] + [merged] + other_remaining)
    return result

def find_partitions_greedy_div(X, thresholds, priors, threshold_true, c, eps=1e-9, show_steps=False):
    n = len(priors)
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_conditional(X, block, thresholds, priors, threshold_true, c)
                + evaluate_conditional(X_eps, block, thresholds, priors, threshold_true, c) * eps)

    t0 = time.perf_counter()
    partition_0 = [list(range(n))]
    acc_loss_0 = block_cost(partition_0[0])
    pq = [(acc_loss_0, partition_0)]

    while pq:
        acc_loss_merged, partition_merged = heapq.heappop(pq)
        if len(partition_merged[0])==1:
            loss = evaluate_system(X, partition_merged, thresholds, priors, threshold_true, c)
            return partition_merged, loss, time.perf_counter()-t0
        partitions = split_partition(partition_merged)
        pq = []
        for partition in partitions:
            acc_loss_split = 0.
            for block in partition:
                acc_loss_split += block_cost(block) * np.sum(priors[block])
            gain = acc_loss_merged - acc_loss_split
            if gain > -1e-9:
                heapq.heappush(pq, (acc_loss_split, partition))
        if not pq:
            loss = evaluate_system(X, partition_merged, thresholds, priors, threshold_true, c)
            return partition_merged, loss, time.perf_counter()-t0

def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [21]:
def validate(priors, thresholds, tt, c):
    p = np.asarray(priors, dtype=float)
    t = np.asarray(thresholds, dtype=float)
    if p.shape != t.shape or p.ndim != 1:
        raise ValueError("priors and thresholds must be 1-D of equal length")
    if np.any(p < 0):
        raise ValueError("priors must be nonnegative")
    if not np.isclose(p.sum(), 1.0):
        raise ValueError(f"priors must sum 1, got {p.sum()}")
    if c <= 0:
        raise ValueError("c must be positive")
    
    return p, t, float(tt), float(c)

def accept_matrix(thresholds):
    t = np.asarray(thresholds, dtype=float)
    return (t[None, :] >= t[:, None]).astype(float)

def build_grid(m, lo=0.0, hi=1.0):
    X = np.linspace(lo, hi, m)
    D = np.full(m, 1.0/m)
    return X, D

def build_constants(priors, thresholds, tt, c, m, eps=1e-6):
    priors, thresholds, tt, c = validate(priors, thresholds, tt, c)
    n = len(thresholds)
    X, D = build_grid(m)

    A = np.empty((m, n+1))
    A[:, 0] = X
    A[:, 1:] = thresholds[None, :]

    cost = c * np.abs(A - X[:, None])
    Hx = (A[:, None, :] >= thresholds[None, :, None]).astype(float)
    U = priors[None, :, None] * (Hx - (1.0 + eps) * cost[:, None, :])
    f = (X >= tt).astype(float)
    L = np.where(f[:, None, None] == 0.0, Hx, 1.0 - Hx)
    M = 1.0 + (1.0 + eps) * cost.max(axis=1)

    valid = A >= X[:, None] - 1e-12
    for xi in range(m):
        seen = set()
        for a in range(n+1):
            key = round(float(A[xi, a]), 12)
            if key in seen:
                valid[xi, a] = False
            else:
                seen.add(key)
    return dict(p=priors, t=thresholds, tt=tt, c=c, n=n, m=m, X=X, D=D, A=A, cost=cost, Hx=Hx, U=U, L=L, M=M, eps=eps, valid=valid)

In [22]:
def build_variables(K):
    n, m = K["n"], K["m"]

    xv = cp.Variable((n, n), boolean=True, name="x")
    yv = cp.Variable((m*n, n+1), boolean=True, name="y")

    cons = [
        cp.sum(xv, axis=1) == 1,
        cp.sum(yv, axis=1) == 1,
    ]
    return xv, yv, cons

def row(x_idx, j, n):
    return x_idx * n + j

def assignment_to_partition(assign):
    bins = {}
    for i, j in enumerate(assign):
        bins.setdefault(j, []).append(i)
    return frozenset(frozenset(v) for v in bins.values())

def bell(n):
    r = [1]
    for _ in range(n):
        new = [r[-1]]
        for v in r:
            new.append(new[-1] + v)
        r = new
    return r[0]

In [23]:
def add_best_response(K, xv, yv):
    n, m, U, M, valid = K["n"], K["m"], K["U"], K["M"], K["valid"]
    ones = np.ones((1, n+1))
    cons = []

    for x_idx in range(m):
        util = xv.T @ U[x_idx]
        y_x = yv[row(x_idx, 0, n): row(x_idx, 0, n)+n, :]

        for a in range(n+1):
            if not valid[x_idx, a]:
                cons.append(y_x[:, a] == 0)
                continue
            u_a = cp.reshape(util[:, a], (n,1), order="C")
            y_a = cp.reshape(y_x[:, a], (n,1), order="C")
            cons.append((u_a + M[x_idx] * (1 - y_a)) @ ones >= util)
    return cons

def chosen_landing(K, yv_value):
    n, m = K["n"], K["m"]
    Y = np.asarray(yv_value).reshape(m, n, n+1)
    a = Y.argmax(axis=2)
    return K["A"][np.arange(m)[:, None], a]

def brute_force_landing(K, assign):
    n, m = K["n"], K["m"]
    out = np.zeros((m, n))
    for j in range(n):
        B = (assign == j).astype(float)
        if B.sum() == 0:
            out[:, j] = np.nan
            continue
        score = (K["U"] + B[None, :, None]).sum(axis=1)
        score = np.where(K["valid"], score, -np.inf)
        out[:, j] = K["A"][np.arange(m), score.argmax(axis=1)]
    return out

In [ ]:
def add_objective(K, xv, yv):
    n, m , L, D, p = K["n"], K["m"], K["L"], K["D"], K["p"]

    v = cp.Variable((m * n, n), nonneg=True, name="v")
    cons = []
    for x_idx in range(m):
        sl = slice(x_idx * n, (x_idx + 1) * n)
        g = yv[sl, :] @ L[x_idx].T
        cons.append(v[sl, :] >= g + xv.T - 1)

    W = np.repeat(D, n)[:, None] * p[None, :]
    return cp.sum(cp.multiply(W, v)), cons

def evaluate_partition(K, assign):
    n, m, U, L, D, p, valid = K["n"], K["m"], K["U"], K["L"], K["D"], K["p"], K["valid"]

    total = 0.0
    for j in range(n):
        members = np.flatnonzero(assign == j)
        if members.size == 0:
            continue
        B = np.zeros(n)
        B[members] = 1.0
        score = (U * B[None, :, None]).sum(axis=1)
        a_star = np.where(valid, score, -np.inf).argmax(axis=1)
        err = L[np.arange(m)[:, None], members[None, :], a_star[:, None]]
        total += float(D @ (err @ p[members]))
    return total

def add_symmetry(K, xv):
    n = K["n"]
    cons = [xv[i, j] == 0 for i in range(n) for j in range(i+1, n)]
    for i in range(1, n):
        for j in range(1, i+1):
            cons.append(xv[i,j] <= cp.sum(xv[:i, j-1]))
    return cons

def pin_empty_bins(K, xv, yv):
    n = K["n"]
    occ = cp.sum(xv, axis=0)
    return [yv[x_idx * n:(x_idx + 1) * n, 0] >= 1 - occ for x_idx in range(K["m"])]

def solve(priors, thresholds, tt, c, m=51, eps=1e-6, prune=True, pin_empty=True, feas_tol=1e-9, check_tol=1e-9, time_limit=None, solver=cp.GUROBI):
    K = build_constants(priors, thresholds, tt, c, m, eps)
    xv, yv, cons = build_variables(K)
    cons = cons + add_best_response(K, xv, yv)
    if prune:
        cons = cons + add_symmetry(K, xv)
    if pin_empty:
        cons = cons + pin_empty_bins(K, xv, yv)
    obj, obj_cons = add_objective(K, xv, yv)

    prob = cp.Problem(cp.Minimize(obj), cons + obj_cons)
    if str(solver).upper() == 'GUROBI':
        kw = {"FeasibilityTol": feas_tol, "IntFeasTol": feas_tol}
        if time_limit is not None:
            kw["TimeLimit"] = time_limit
    else:
        kw = dict(primal_feasibility_tolerance=feas_tol, mip_feasibility_tolerance=feas_tol)
        if time_limit is not None:
            kw["time_limit"] = time_limit
    
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob.solve(solver=solver, **kw)
    elapsed = time.perf_counter() - t0

    assign = np.asarray(xv.value).argmax(axis=1)
    loss = evaluate_partition(K, assign)
    return dict(
        status=prob.status,
        partition=sorted(sorted(b) for b in assignment_to_partition(assign)),
        loss=loss,
        solver_objective=float(prob.value),
        certified=bool(loss - float(prob.value) < check_tol),
        seconds=elapsed,
        K=K
    )

def brute_force(K):
    t0 = time.perf_counter()
    n = K["n"]
    best = (np.inf, None)
    for assign in itertools.product(range(n), repeat=n):
        val = evaluate_partition(K, np.array(assign))
        if val < best[0] - 1e-12:
            best = (val, np.array(assign))
    return best, time.perf_counter() - t0

In [27]:
cp.GUROBI

'GUROBI'

In [25]:
def add_result(results, alg, time, loss, ratio, partition, priors, thresholds, c, tt, m, n):
    results["alg"].append(alg)
    results["time"].append(time)
    results["loss"].append(loss)
    results["ratio"].append(ratio)
    results["partition"].append(partition)
    results["priors"].append(priors)
    results["thresholds"].append(thresholds)
    results["c"].append(c)
    results["tt"].append(tt)
    results["m"].append(m)
    results["n"].append(n)

def run_example(priors, thresholds, tt, c, m, run_unreduced=True, run_solver=True, time_limit=None):
    n = len(thresholds)
    K = build_constants(priors, thresholds, tt, c, m)

    results = {"alg": [], "time": [], "loss": [], "ratio": [], "partition": [], "priors": [], "thresholds": [], "c": [], "tt": [], "m": [], "n": []}
    p_agg, loss_agg, time_agg = find_partitions_greedy_agg(K["X"], thresholds, priors, tt, c)
    p_div, loss_div, time_div = find_partitions_greedy_div(K["X"], thresholds, priors, tt, c)
    p_opt, loss_opt, time_opt = find_partitions_optimal(K["X"], thresholds, priors, tt, c)
    
    r_agg  = approximation_ratio(loss_opt, loss_agg)
    r_div  = approximation_ratio(loss_opt, loss_div)

    print(f"m                  : {m}")
    print(f"n                  : {n}")
    print(f"c                  : {c}")
    print(f"t*                 : {tt:.4f}")
    with np.printoptions(formatter={'float': '{: 0.4f}'.format}):
        print(f"priors             : {priors}")
        print(f"thresholds         : {thresholds}\n")

    print(f"[Alg: OPT       ]  |  Time: {time_opt:7.2f}s  |  Loss: {loss_opt:7.3f}  |  Ratio: {1.0:7.3f}  |  Partition: {p_opt}")
    print(f"[Alg: AGG       ]  |  Time: {time_agg:7.2f}s  |  Loss: {loss_agg:7.3f}  |  Ratio: {r_agg:7.3f}  |  Partition: {p_agg}")
    print(f"[Alg: DIV       ]  |  Time: {time_div:7.2f}s  |  Loss: {loss_div:7.3f}  |  Ratio: {r_div:7.3f}  |  Partition: {p_div}")
    
    add_result(results, "OPT", time_opt, loss_opt, 1.0, p_opt, priors, thresholds, c, tt, m, n)
    add_result(results, "AGG", time_agg, loss_agg, r_agg, p_agg, priors, thresholds, c, tt, m, n)
    add_result(results, "DIV", time_div, loss_div, r_div, p_div, priors, thresholds, c, tt, m, n)
    
    if run_solver:
        mip_algs = [("MILP_R", True)]
        if run_unreduced:
            mip_algs.append(("MILP", False))
        for name_, prune in mip_algs:
            milp = solve(priors, thresholds, tt, c, m, prune=prune)
            p_milp, loss_milp, time_milp = milp["partition"], milp["loss"], milp["seconds"]
            r_milp = approximation_ratio(loss_opt, loss_milp)
            name = f"Alg: {name_}"
            print(f"[{name:15}]  |  Time: {time_milp:7.2f}s  |  Loss: {loss_milp:7.3f}  |  Ratio: {r_milp:7.3f}  |  Partition: {p_milp}")
            add_result(results, name_, time_milp, loss_milp, r_milp, p_milp, priors, thresholds, c, tt, m, n)

            if time_limit:
                if isinstance(time_limit, float) or isinstance(time_limit, int):
                    milp_t = solve(priors, thresholds, tt, c, m, prune=True, time_limit=time_limit)
                    p_milp_t, loss_milp_t, time_milp_t = milp_t["partition"], milp_t["loss"], milp_t["seconds"]
                    r_milp_t = approximation_ratio(loss_opt, loss_milp_t)
                    print(f"[Alg: MILP-{time_limit}s]  |  Time: {time_milp_t:7.2f}s  | Loss: {loss_milp_t:7.3f}  |  Ratio: {r_milp_t:7.3f}  |  Partition: {p_milp_t}")
                elif isinstance(time_limit, list):
                    for tl in time_limit:
                        milp_t = solve(priors, thresholds, tt, c, m, prune=True, time_limit=tl)
                        p_milp_t, loss_milp_t, time_milp_t = milp_t["partition"], milp_t["loss"], milp_t["seconds"]
                        r_milp_t = approximation_ratio(loss_opt, loss_milp_t)
                        name = f"Alg: {name_}-{tl}s"
                        print(f"[{name:15}]  |  Time: {time_milp_t:7.2f}s  |  Loss: {loss_milp_t:7.3f}  |  Ratio: {r_milp_t:7.3f}  |  Partition: {p_milp_t}")
                        add_result(results, f"{name_}-{tl}s", time_milp_t, loss_milp_t, r_milp_t, p_milp_t, priors, thresholds, c, tt, m, n)
    return results

### Example 1

In [33]:
m = 21
n = 9
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, time_limit=[2, 5, 15], run_unreduced=True)
pd.DataFrame(res1).to_pickle("milp_v_greedy1.pkl")

m                  : 21
n                  : 9
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111]
thresholds         : [ 0.0000  0.1250  0.2500  0.3750  0.5000  0.6250  0.7500  0.8750  1.0000]

[Alg: OPT       ]  |  Time:    2.50s  |  Loss:   0.042  |  Ratio:   1.000  |  Partition: [[1, 2], [0, 3, 4, 6, 7], [5, 8]]
[Alg: AGG       ]  |  Time:    0.02s  |  Loss:   0.053  |  Ratio:   1.250  |  Partition: [[2], [0, 4, 6], [1, 5], [3, 7, 8]]
[Alg: DIV       ]  |  Time:    0.02s  |  Loss:   0.058  |  Ratio:   1.375  |  Partition: [[0, 2, 3, 4, 5, 6, 7], [1], [8]]
[Alg: MILP_R    ]  |  Time:   17.90s  |  Loss:   0.042  |  Ratio:   1.000  |  Partition: [[0, 3, 4, 6, 7], [1, 5], [2], [8]]
[Alg: MILP_R-2s ]  |  Time:    2.24s  |  Loss:   0.090  |  Ratio:   2.125  |  Partition: [[0], [1], [2], [3], [4], [5], [6], [7], [8]]
[Alg: MILP_R-5s ]  |  Time:    5.25s  |  Loss:   0.063  |  Ratio:   1.500  |  

In [26]:
m = 21
n = 9
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, time_limit=[2, 5, 15], run_unreduced=True)
# pd.DataFrame(res1).to_pickle("milp_v_greedy1.pkl")

m                  : 21
n                  : 9
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111]
thresholds         : [ 0.0000  0.1250  0.2500  0.3750  0.5000  0.6250  0.7500  0.8750  1.0000]

[Alg: OPT       ]  |  Time:    2.49s  |  Loss:   0.042  |  Ratio:   1.000  |  Partition: [[1, 2], [0, 3, 4, 6, 7], [5, 8]]
[Alg: AGG       ]  |  Time:    0.02s  |  Loss:   0.053  |  Ratio:   1.250  |  Partition: [[2], [0, 4, 6], [1, 5], [3, 7, 8]]
[Alg: DIV       ]  |  Time:    0.02s  |  Loss:   0.058  |  Ratio:   1.375  |  Partition: [[0, 2, 3, 4, 5, 6, 7], [1], [8]]
Set parameter Username
Set parameter LicenseID to value 2853105
Academic license - for non-commercial use only - expires 2027-08-12


GurobiError: Unknown parameter 'primal_feasibility_tolerance'

### Example 2

In [20]:
m = 501
n = 5
p1 = 0.02/0.98
p2 = 0.02/0.98
diff = 1 - p1 - p2
priors = np.array([0.14 * diff/0.94, p1, p2, 0.56 * diff/0.94, 0.24 * diff/0.94])
thresholds = np.linspace(0.0, 1.0, n)
c = 1.
tt = 0.1489
assert(abs(np.sum(priors) - 1) < 1e-6)

res2 = run_example(priors, thresholds, tt, c, m, time_limit=[2,5,15,30,45])
pd.DataFrame(res2).to_pickle("milp_v_greedy2.pkl")

m                  : 501
n                  : 5
c                  : 1.0
t*                 : 0.1489
priors             : [ 0.1429  0.0204  0.0204  0.5714  0.2449]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

[Alg: OPT       ]  |  Time:    0.01s  |  Loss:   0.027  |  Ratio:   1.000  |  Partition: [[1, 2], [0, 3, 4]]
[Alg: AGG       ]  |  Time:    0.01s  |  Loss:   0.145  |  Ratio:   5.290  |  Partition: [[0, 1], [2, 3, 4]]
[Alg: DIV       ]  |  Time:    0.01s  |  Loss:   0.145  |  Ratio:   5.290  |  Partition: [[2, 3, 4], [0, 1]]
[Alg: MILP_R    ]  |  Time:   57.46s  |  Loss:   0.027  |  Ratio:   1.000  |  Partition: [[0, 3, 4], [1, 2]]
[Alg: MILP_R-2s ]  |  Time:   10.80s  |  Loss:   0.177  |  Ratio:   6.431  |  Partition: [[0, 1, 2, 3, 4]]
[Alg: MILP_R-5s ]  |  Time:   13.87s  |  Loss:   0.177  |  Ratio:   6.431  |  Partition: [[0, 1, 2, 3, 4]]
[Alg: MILP_R-15s]  |  Time:   24.39s  |  Loss:   0.177  |  Ratio:   6.431  |  Partition: [[0, 1, 2, 3, 4]]
[Alg: MILP_R-30

### Example 3

In [10]:
m = 501
n = 5
p = (0.2444/0.25 - 0.97) / (0.4888/0.25 - 0.97)
diff = 1-2*p
priors = np.array([0.097*diff/0.97, p, p, 0.6286*diff/0.97, 0.2444*diff/0.97])
thresholds = np.linspace(0.0, 1.0, n)
c = 1.
tt = 0.1

res3 = run_example(priors, thresholds, tt, c, m, run_solver=True, run_unreduced=True, time_limit=[30, 60, 90, 120])
pd.DataFrame(res3).to_pickle("milp_v_greedy3.pkl")

m                  : 501
n                  : 5
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.0985  0.0077  0.0077  0.6380  0.2481]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

[Alg: OPT       ]  |  Time:    0.01s  |  Loss:   0.013  |  Ratio:   1.000  |  Partition: [[1, 2], [0, 3, 4]]
[Alg: AGG       ]  |  Time:    0.01s  |  Loss:   0.013  |  Ratio:   1.000  |  Partition: [[0, 3, 4], [1, 2]]
[Alg: DIV       ]  |  Time:    0.01s  |  Loss:   0.098  |  Ratio:   7.450  |  Partition: [[2, 3, 4], [0, 1]]
[Alg: MILP_R    ]  |  Time:   68.62s  |  Loss:   0.013  |  Ratio:   1.000  |  Partition: [[0, 3, 4], [1], [2]]
[Alg: MILP_R-30s]  |  Time:   38.77s  |  Loss:   0.173  |  Ratio:  13.137  |  Partition: [[0, 1, 2, 3, 4]]
[Alg: MILP_R-60s]  |  Time:   69.14s  |  Loss:   0.172  |  Ratio:  13.096  |  Partition: [[0, 1, 3, 4], [2]]
[Alg: MILP_R-90s]  |  Time:   69.59s  |  Loss:   0.013  |  Ratio:   1.000  |  Partition: [[0, 3, 4], [1], [2]]
[Alg: M

### Random Examples

In [22]:
def add_rand_result(results, m, n, i, alg, time, loss, partition, priors, thresholds, c, tt):
    results["m"].append(m)
    results["n"].append(n)
    results["i"].append(i)
    results["alg"].append(alg)
    results["time"].append(time)
    results["loss"].append(loss)
    results["partition"].append(partition)
    results["priors"].append(priors)
    results["thresholds"].append(thresholds)
    results["c"].append(c)
    results["tt"].append(tt)

In [ ]:
rng = np.random.default_rng(0)
m = 101
N = [7, 8, 9, 10]

for n in N:
    results_rand = {"m": [], "n": [], "i": [], "alg": [], "time": [], "loss": [], "partition": [], "priors": [], "thresholds": [], "c": [], "tt": []}
    K = build_constants(priors, thresholds, tt, c, m)
    for i in tqdm.trange(100, desc=f"[n={n}]"):
        priors = rng.dirichlet(np.ones(n))
        thresholds = np.sort(rng.uniform(0, 1, n))

        c = rng.uniform(0.0, 2.0)
        tt = rng.uniform(0.0, 1.0)

        mip = solve(priors, thresholds, tt, c, m, prune=True)
        p_mip, loss_mip, time_mip = mip["partition"], mip["loss"], mip["seconds"]
        p_agg, loss_agg, time_agg = find_partitions_greedy_agg(K["X"], thresholds, priors, tt, c)
        p_div, loss_div, time_div = find_partitions_greedy_div(K["X"], thresholds, priors, tt, c)

        add_rand_result(results_rand, m, n, i, "MILP_R", time_mip, loss_mip, p_mip, priors, thresholds, c, tt)
        add_rand_result(results_rand, m, n, i, "AGG", time_agg, loss_agg, p_agg, priors, thresholds, c, tt)
        add_rand_result(results_rand, m, n, i, "DIV", time_div, loss_div, p_div, priors, thresholds, c, tt)
    df = pd.DataFrame(results_rand)
    df.to_pickle(f"rand_examples_solver_greedy_m{m}_n{n}.pkl") 

[n=8]:  79%|███████▉  | 79/100 [2:56:14<1:08:02, 194.39s/it]

In [20]:
def generate_prior_grid(n_components, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls - n_components):
        counts = np.bincount(combo, minlength=n_components) + 1
        grids.append((counts * step).round(4))
    return grids

In [21]:
M = [11, 21]
N = [4, 5, 6]
tt = 0.1
c = 1.0
results = {"alg": [], "time": [], "loss": [], "ratio": [], "partition": [], "priors": [], "thresholds": [], "c": [], "tt": [], "m": [], "n": []}
for m in M:
    for n in N:
        results = {"i": [], "alg": [], "time": [], "loss": [], "partition": []}
        thresholds = np.linspace(0.0, 1.0, n)
        prior_grid = generate_prior_grid(n, 20)
        for i, priors in tqdm.tqdm(enumerate(prior_grid), desc=f"[m={m}] [n={n}]", total=len(prior_grid)):
            K = build_constants(priors, thresholds, tt, c, m)

            p_agg, loss_agg, time_agg = find_partitions_greedy_agg(K["X"], thresholds, priors, tt, c)
            p_div, loss_div, time_div = find_partitions_greedy_div(K["X"], thresholds, priors, tt, c)
            results["i"].append(i)
            results["alg"].append("agg")
            results["time"].append(time_agg)
            results["loss"].append(loss_agg)
            results["partition"].append(p_agg)
            
            results["i"].append(i)
            results["alg"].append("div")
            results["time"].append(time_div)
            results["loss"].append(loss_div)
            results["partition"].append(p_div)
        
        df = pd.DataFrame(results)
        df.to_pickle(f"grid_search_greedy_m{m}_n{n}.pkl")

[m=21] [n=6]: 100%|██████████| 11628/11628 [03:51<00:00, 50.29it/s]
